In [19]:
import os
from pypdf import PdfReader
import ollama
import chromadb
import uuid

In [20]:
#load all content from pdf file into a document variable
def load_pdfs(folder_path):
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            filepath = os.path.join(folder_path, filename)
            reader = PdfReader(filepath)
            text = ""
            for page in reader.pages:
                text += page.extract_text()
            documents.append({"filename":filename, "text":text})
            print(f"Loaded: {filename}")
    return documents

docs_folder = r"C:/Users/Asus/OneDrive/Desktop/rag-project/docs"
documents = load_pdfs(docs_folder)
print(f"\nTotal documents loaded: {len(documents)}")


Loaded: attention_is_all_you_need.pdf
Loaded: bert.pdf
Loaded: gpt3.pdf
Loaded: rag.pdf
Loaded: wordtovec.pdf

Total documents loaded: 5


In [21]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0 #first text character
    while start < len(text):
        end = start + chunk_size # 0+500 = 500 characters || 450+500 = 950 characters etc.
        chunks.append(text[start:end])
        start = start + (chunk_size - overlap)  #0+500-50 = 450 characters
    return chunks

for doc in documents:
    chunks = chunk_text(doc["text"])
    print(f"{doc['filename']}: {len(chunks)} chunks")


attention_is_all_you_need.pdf: 50 chunks
bert.pdf: 81 chunks
gpt3.pdf: 296 chunks
rag.pdf: 87 chunks
wordtovec.pdf: 49 chunks


In [22]:
def embed_chunks(chunks):
    response = ollama.embed(
        model='nomic-embed-text', 
        input=chunks
        )
    
    return response['embeddings']

In [23]:
chroma_client = chromadb.PersistentClient(path="chroma_db")
collection = chroma_client.get_or_create_collection(name="research_papers")

#now, we add data chunks and embeddings onto chroma db
if collection.count() == 0:
    for doc in documents:
        chunks = chunk_text(doc["text"])
        embeddings = embed_chunks(chunks)
        collection.add(
            documents = chunks,
            embeddings = embeddings,
            metadatas = [{"source":  doc["filename"]} for _ in chunks],
            ids = [str(uuid.uuid4()) for _ in chunks]
        )
        
        print(f"Added {len(chunks)} chunks from {doc['filename']}")
else:
    print(f"Collection already has {collection.count()} chunks, skipping ingestion")

Collection already has 999 chunks, skipping ingestion


In [24]:
question = input("Ask a query: ")

question_embeddings = embed_chunks([question])[0]
results = collection.query(
    query_embeddings = [question_embeddings],
    n_results = 5,
)

context = "\n\n".join(results["documents"][0])

system_instruction = """You are a helpful research assistant. 
Answer the user's question using ONLY the provided context.
Explain clearly and thoroughly, as if teaching a beginner.
Use examples where possible. Be detailed and comprehensive.
Stay strictly focused on what the user asked — do not mix information from unrelated topics.
If the context doesn't contain enough information to answer, say so clearly.
Format your response in exactly 2 paragraphs. Each paragraph must be at least 3 sentences long."""
user_prompt = f"Context: {context}\n Question: {question}"

sources = [m["source"] for m in results["metadatas"][0]]
print(f"Sources used: {sources}")
response = ollama.chat(
    model = "llama3.1:8b",
    messages = [
        {"role":"system", "content":system_instruction},
        {"role":"user", "content":user_prompt}
    ],
)

print(response["message"]["content"])


Sources used: ['rag.pdf', 'wordtovec.pdf', 'gpt3.pdf', 'gpt3.pdf', 'rag.pdf']
Let's break down the context to understand what it's about. We have two main parts here. First, we have a research paper titled "SuperGLUE: A Stickier Benchmark for General-Purpose Language Understanding Systems" by Omer Levy and Samuel Bowman. Then, there is another section that provides examples of sentences with made-up words like Burringo, Gigamuru, and screeg. This section seems to be testing the understanding of language models on recognizing these words and generating accurate responses.

Now, let's move on to your question: "What is DPR in RAG?" From the context provided, we can see that DPR is mentioned only once, but it's not clearly defined. However, based on the surrounding information, it seems likely that DPR refers to a model or architecture used within the RAG (Rapid Generation of Analogies for Natural Language) framework. Unfortunately, without more specific details in the context, we cannot 

In [25]:
# Eval set
import json
import sys

with open("eval_set.json","r") as f:
    eval_set = json.load(f)

print(f"Loaded {len(eval_set)} eval questions")

correct = 0

for item in eval_set:
    question = item["question"]
    expected_source = item["source"]

    question_embedding = embed_chunks([question])[0]
    results = collection.query(
        query_embeddings = [question_embedding],
        n_results = 5
    )

    sources = [m["source"] for m in results["metadatas"][0]]
    source_correct = expected_source in sources
    if source_correct:
        correct+=1

    print(f"Q: {question}")
    print(f"Expected: {expected_source} | Got: {sources[0]} | {'✓' if source_correct else '✗'}")
    print("---")
    
print(f"\nRetrieval accuracy: {correct}/{len(eval_set)} = {correct/len(eval_set)*100:.1f}%")



#--------------------------------------------------------------------


Loaded 20 eval questions
Q: What optimizer was used to train the Transformer?
Expected: attention_is_all_you_need.pdf | Got: bert.pdf | ✓
---
Q: How many attention heads does the base Transformer use?
Expected: attention_is_all_you_need.pdf | Got: attention_is_all_you_need.pdf | ✓
---
Q: What is the model dimension dmodel in the base Transformer?
Expected: attention_is_all_you_need.pdf | Got: attention_is_all_you_need.pdf | ✓
---
Q: What dataset was used for English to German translation in the Transformer paper?
Expected: attention_is_all_you_need.pdf | Got: gpt3.pdf | ✓
---
Q: What does BERT stand for?
Expected: bert.pdf | Got: gpt3.pdf | ✗
---
Q: What are the two tasks BERT was pre-trained on?
Expected: bert.pdf | Got: bert.pdf | ✓
---
Q: How many parameters does BERT-Large have?
Expected: bert.pdf | Got: rag.pdf | ✓
---
Q: What is the masked language model objective in BERT?
Expected: bert.pdf | Got: bert.pdf | ✓
---
Q: How many parameters does GPT-3 have?
Expected: gpt3.pdf | Got: